In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv("./data/bank-additional-full.csv", sep=";")
target = "y"
data[target] = (data[target] == "yes").astype(int)
exclude = ["duration"]

data["no_previous_contact"] = (data["pdays"] == 999).astype(int)
data["pdays"] = data["pdays"].replace(999, np.nan)

# Стратифікований розподіл даних
train, temp = train_test_split(
    data, test_size=0.4, random_state=42, stratify=data[target]
)

valid, test = train_test_split(
    temp, test_size=0.5, random_state=42, stratify=temp[target]
)

rest = pd.concat([train, valid])

features = [c for c in data.columns if c not in exclude + [target]]
numeric = train[features].select_dtypes("number").columns.tolist()
categorical = [c for c in features if c not in numeric]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")),
     categorical),
])

def metrics(y_true, y_pred):
    return pd.Series({
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }).round(3)

def cost(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return 15 * fnr + fpr

model = make_pipeline(
    pre, LogisticRegression(max_iter=1000))
model.fit(train[features], train[target])

threshold = 0.21          # обрано в ЛР6 на валідаційній частині

final = clone(model)
final.fit(rest[features], rest[target])

y_test = test[target]
test_proba = final.predict_proba(test[features])[:, 1]
test_pred = (test_proba >= threshold).astype(int)
print("Вартість на тесті:", cost(y_test, test_pred))

Вартість на тесті: 6.5248449219302795


Результат збігається з отриманим на ЛР-6.

In [2]:
from pathlib import Path

import joblib

Path("models").mkdir(exist_ok=True)
joblib.dump(final, "models/model.joblib")
print(Path("models/model.joblib").stat().st_size, "байт")

7754 байт


In [3]:
loaded = joblib.load("models/model.joblib")
loaded_proba = loaded.predict_proba(test[features])[:, 1]
loaded_pred = (loaded_proba >= threshold).astype(int)

print("Імовірності збігаються:", np.array_equal(loaded_proba, test_proba))
print("Вартість на тесті:", cost(y_test, loaded_pred))

Імовірності збігаються: True
Вартість на тесті: 6.5248449219302795


Розмір файлу моделі: 7754 байт, перевірено, що результати збігаються.

In [4]:
import json
import platform
from datetime import date

import sklearn

test_metrics = metrics(y_test, test_pred).to_dict()
test_metrics["Вартість"] = int(cost(y_test, test_pred))

metadata = {
    "model": type(final[-1]).__name__,
    "trained_on": "train + valid (rest)",
    "threshold": threshold,
    "cost_formula": "C = 15 * (fn / (fn + tp)) + fp / (fp + tn)",
    "features": features,
    "python": platform.python_version(),
    "scikit-learn": sklearn.__version__,
    "created": date.today().isoformat(),
    "test_metrics": test_metrics,
}

with open("models/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

"model": назва моделі  
"trained_on": на якому наборі даних модель тренується  
"threshold": порогове значення для вибору прогнозу  
"cost_formula": формула вартості  
"features": список назв властивостей  
"python": версія Python що використовувалась для тренування  
"scikit-learn": версія бібліотеки що використовувалась для тренування  
"created": дата тренування моделі  
"test_metrics": метрики моделі натренової на наборі даних  

Тестові метрики можна записати лише після перевірки збігу прогнозів, тому що будь-яка метрика (Accuracy, F1, Cost) є результатом математичного порівняння двох векторів: фактичних відповідей (y_true) та прогнозів моделі (y_pred).
Перед записом метрик треба переконатися, що прогнози архітектурно та логічно збігаються з тестовою вибіркою.

In [5]:
!python train.py

Модель навчено на 32950 записах і збережено в models/


In [6]:
import joblib

new = joblib.load("models/model.joblib")
new_proba = new.predict_proba(test[features])[:, 1]
same = np.array_equal(new_proba, test_proba)
print("Прогнози збігаються з ЛР6:", same)

if same:
    with open("models/metadata.json", encoding="utf-8") as f:
        meta = json.load(f)
    meta["test_metrics"] = test_metrics
    with open("models/metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print("Тестові метрики перенесено в metadata.json")

Прогнози збігаються з ЛР6: True
Тестові метрики перенесено в metadata.json


In [7]:
demo = test.drop(columns=[target]).head(5)
demo.to_csv("new_records.csv", index=False)
print(test_proba[:5].round(3))

[0.077 0.024 0.042 0.046 0.039]


In [8]:
!python predict.py new_records.csv

   probability  decision
0        0.077         0
1        0.024         0
2        0.042         0
3        0.046         0
4        0.039         0


In [9]:
from predict import predict

predict(demo.head(2))

,probability,decision
33978,0.077,0
4006,0.024,0


In [10]:
print("pandas==" + pd.__version__)
print("numpy==" + np.__version__)
print("scikit-learn==" + sklearn.__version__)
print("joblib==" + joblib.__version__)

pandas==3.0.5
numpy==2.5.3
scikit-learn==1.9.0
joblib==1.6.0


Версії scikit-learn збігаються.

високий: додатково — поведінка програми з відсутніми ознаками і невідомими категоріями в розділі «Обмеження».

Проблема. Остаточна модель із ЛР6 існує лише в ноутбуці; щоб отримати прогноз, треба заново виконати всі клітинки. Застосунку потрібна готова модель і спосіб отримати рішення для вхідних записів.

Варіанти. Зберегти лише класифікатор і відтворювати підготовку даних у програмі прогнозування; зберегти весь конвеєр (підготовка + логістична регресія) у файл joblib, а поріг і опис — в окремий файл.

Рішення. Весь конвеєр збережено в models/model.joblib, поріг 0,21, перелік ознак, версії і тестові метрики — у models/metadata.json. Модель відтворюється скриптом train.py, прогнози видає функція predict зі скрипту predict.py.

Обґрунтування. Збереження конвеєра цілком виключає розбіжність підготовки даних між навчанням і застосуванням. Завантажена модель і модель зі скрипту дають на тесті ті самі ймовірності, що й модель ЛР6 (точний збіг), тому тестові метрики ЛР6 належать збереженій моделі.

Наслідки. Прогноз отримується однією командою без ноутбука; функцію predict можна викликати з API в ЛР8. Для запуску потрібні версії бібліотек із requirements.txt.

Обмеження. Файл моделі прив’язаний до scikit-learn 1.9.0. Файл joblib можна завантажувати лише з надійного джерела. Для відсутніх ознак програма видає зрозуміле повідомлення про помилку, але для невідомої категорії мовчки видає прогноз: відділ, якого не було в даних, змінює ймовірність без попередження.


In [11]:
bad = demo.drop(columns=["age", "job"])
predict(bad)

ValueError: Бракує ознак: age, job

In [14]:
odd = demo.head(1).copy()
odd["Department"] = "Legal"
predict(pd.concat([demo.head(1), odd]))

,probability,decision
33978,0.077,0
33978,0.077,0


Для невідомої категорії всі стовпці цієї ознаки після кодування дорівнюють нулю, тобто модель поводиться так, ніби ознака не належить жодній відомій категорії через OneHotEncoder(handle_unknown="ignore").

**Проблема.** Остаточна модель із ЛР6 існує лише в ноутбуці; щоб отримати прогноз, треба заново виконати всі клітинки. Застосунку потрібна готова модель і спосіб отримати рішення для вхідних записів.  

**Варіанти.** Зберегти лише класифікатор і відтворювати підготовку даних у програмі прогнозування; зберегти весь конвеєр (підготовка + логістична регресія) у файл joblib, а поріг і опис — в окремий файл.  

**Рішення.** Весь конвеєр збережено в models/model.joblib, поріг 0,21, перелік ознак, версії і тестові метрики — у models/metadata.json. Модель відтворюється скриптом train.py, прогнози видає функція predict зі скрипту predict.py.  

**Обґрунтування.** Збереження конвеєра цілком виключає розбіжність підготовки даних між навчанням і застосуванням. Завантажена модель і модель зі скрипту дають на тесті ті самі ймовірності, що й модель ЛР6 (точний збіг), тому тестові метрики ЛР6 належать збереженій моделі.  

**Наслідки.** Прогноз отримується однією командою без ноутбука; функцію predict можна викликати з API в ЛР8. Для запуску потрібні версії бібліотек із requirements.txt.  

**Обмеження.** Файл моделі прив’язаний до scikit-learn 1.9.0. Файл joblib можна завантажувати лише з надійного джерела.

У випадку відсутніх ознак програма видає помилку з переліком відсутніх ознак. Для невідомої категорії мовчки видає прогноз, не змінює ймовірність.